# LoRA Lab: Parameter-Efficient Fine-Tuning of a Language Model

## Introduction

Pretrained large language models are extraordinary general-purpose learners, but out of the box they rarely speak in the exact *style*, *format*, or *domain* you need. The obvious fix is to **fine-tune** — keep training the model on task-specific data. The problem is that modern LLMs have billions of parameters, and updating all of them is expensive in memory, compute, *and* storage: every new task gives you another multi-gigabyte checkpoint to manage.

**LoRA** (Low-Rank Adaptation; Hu et al., 2021) sidesteps this by leaving the pretrained weights *frozen* and inserting tiny trainable "adapter" matrices into the model. You train only a few million parameters instead of a few billion, and the resulting adapter is small enough to email.

### The LoRA reparameterization

Consider a single linear layer in a transformer: $y = W x$, with $W \in \mathbb{R}^{d_\text{out} \times d_\text{in}}$. Full fine-tuning would update $W$ directly, giving $W + \Delta W$ with $\Delta W$ of the same (huge) shape.

LoRA's key assumption is that the *update* $\Delta W$ has low **intrinsic rank** — even though $W$ itself is high-dimensional, the task-specific adjustment lies in a much smaller subspace. We enforce this by parameterizing the update as a product of two thin matrices:

$$\Delta W = B A, \qquad A \in \mathbb{R}^{r \times d_\text{in}}, \quad B \in \mathbb{R}^{d_\text{out} \times r}, \quad r \ll \min(d_\text{in}, d_\text{out})$$

The forward pass becomes

$$y = W x + \frac{\alpha}{r}\, B A x$$

where $\alpha$ is a fixed scaling constant. At initialization, $A$ is random and $B = 0$, so the adapter starts as a no-op and the model reproduces the base behavior exactly. Training updates only $A$ and $B$; the original $W$ is frozen.

Parameter count: instead of $d_\text{out} \cdot d_\text{in}$ trainable parameters per layer, you have $r \cdot (d_\text{in} + d_\text{out})$ — typically a **reduction of 100–10000×**.

### What we'll build

In this lab you will fine-tune [TinyLlama-1.1B-Chat](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) to rewrite ordinary English sentences in exaggerated *pirate speak* ("Belay fer yer mateys in th' tavern, matey"). Pirate translation is a toy task, but it has one very useful property: the base model is *bad* at it (it knows what pirates sound like in a vague way, but it won't commit to the style), so any improvement from fine-tuning is immediately visible in the output.

You will:

1. **Inspect the dataset** — a synthetic instruction/input/output corpus
2. **Format examples** with a chat template and mask prompt tokens so loss applies only to the response
3. **Load TinyLlama** and generate a *baseline* (un-tuned) pirate translation
4. **Inject LoRA adapters** via the `peft` library and count trainable parameters
5. **Train** for a couple of epochs with the HuggingFace `Trainer`
6. **Compare** base vs. LoRA outputs on held-out prompts
7. **Sweep rank $r$** to see the capacity/overfitting tradeoff

### Simplifications

To keep this lab tractable on modest hardware we make several compromises relative to real fine-tuning jobs:

| This lab | Production fine-tuning |
|----------|------------------------|
| 2000 synthetic training examples | Millions of curated, human-filtered examples |
| TinyLlama 1.1B | Llama 70B, Mixtral, Qwen2-72B, etc. |
| Single LoRA rank, no sweep on held-out task metric | Hyperparameter search over $r$, $\alpha$, dropout, target modules |
| LoRA on attention projections only | LoRA + DoRA + QLoRA on all linear layers, sometimes embedding layers too |
| Loss-only evaluation | Task-specific metrics (BLEU, pass@k, win-rate vs. baseline, human eval) |
| Single-GPU or CPU | Multi-node distributed training with DeepSpeed/FSDP |


## 0. Setup

We need `transformers`, `peft`, `accelerate`, `datasets`, and `torch`. If you're running this on Colab or a fresh environment, uncomment and run the install cell.


In [ ]:
# !pip install -q "transformers>=4.40" "peft>=0.10" "accelerate>=0.29" datasets torch

import json
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

DEVICE = cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


## 1. The Dataset: Instruction / Input / Output

Our training data lives in two JSON-Lines files (`pirate_train.jsonl` and `pirate_val.jsonl`). Each line is a single example with three fields:

- **`instruction`** — the task description ("Translate this sentence into pirate speech.")
- **`input`** — the sentence to be rewritten
- **`output`** — the target pirate-style response

This is the standard **instruction-tuning** format popularized by the Alpaca dataset. The model learns to map `(instruction, input) → output`; at inference time we provide the first two and ask the model to produce the third.

Two important properties of our data:

1. **It is synthetic.** Every example was generated by programmatically substituting words from a small vocabulary into templates. This means the data is internally consistent but has *limited diversity* — a point we'll return to in the exercises.
2. **The output style is highly regular.** A handful of substitutions (`the → th'`, `your → yer`, `friend → matey`, `bar → tavern`, `will → will`, `-ing → -in'`) account for almost all of it. This is exactly the kind of task LoRA excels at: a small, low-rank stylistic shift applied on top of an already-competent language model.


In [ ]:
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl("pirate_train.jsonl")
val_rows   = read_jsonl("pirate_val.jsonl")

print(f"Train examples: {len(train_rows)}")
print(f"Val examples:   {len(val_rows)}")
print()
print("Three random training examples:")
for ex in random.sample(train_rows, 3):
    print(f"  instruction: {ex['instruction']}")
    print(f"  input:       {ex['input']}")
    print(f"  output:      {ex['output']}")
    print()


### Exercise 1: Inspecting the data

**1a.** Count how many *unique* `instruction` strings appear in the training set. How many unique `output` strings? Given that there are 2000 examples, what does this tell you about the diversity of the dataset?

**1b.** *(Conceptual)* Our dataset was generated by filling slots in a small set of templates. Suppose instead we had 2000 *real* human-written pirate translations scraped from novels. Which dataset would you expect to generalize better to the prompt `"I need to return the book I borrowed from the library."` — and why? Think about what kinds of words and grammatical structures templated data does and does not cover.

**1c.** *(Conceptual)* Why do we hold out a separate validation set at all? In a purely synthetic dataset generated from the same templates, train and validation examples are essentially interchangeable. What is the `eval_loss` on such a val set actually measuring, and what is it *not* measuring?


In [ ]:
# Exercise 1 — your code here


## 2. Formatting Examples: Chat Templates and Masked Loss

Before we can feed anything to the model, we need to turn each `(instruction, input, output)` triple into a single token sequence. Two subtle choices matter here:

### 2.1 Chat templates

TinyLlama-Chat is an **instruction-tuned** model: during its own pretraining / SFT, it saw text wrapped in special role markers (`<|user|>`, `<|assistant|>`, and so on). If we feed it plain text at fine-tune time, we'd be fighting its own prior. The tokenizer ships with a `chat_template` that knows how to render messages in the exact format the model expects — we just hand it a list of `{"role": ..., "content": ...}` dicts.

### 2.2 Masking the prompt from the loss

A causal language model is trained with the next-token prediction loss

$$\mathcal{L} = -\sum_{t=1}^{T} \log P_\theta(y_t \mid y_{<t})$$

If we compute this loss over the entire prompt-plus-response sequence, the model gets gradient signal for predicting the *prompt tokens* too — tokens like "`Translate this sentence into pirate speech.`" that the user will always type in and that the model should never be in the business of generating. That wastes capacity and can actively hurt performance.

The fix is simple: build the full sequence, tokenize it, then set the label for every prompt-position token to `-100`. HuggingFace's loss function skips any position whose label is `-100`, so those tokens contribute to attention (the model still *sees* them) but not to the gradient. Only the response tokens are actually supervised.

$$\ell_t = \begin{cases} -\log P_\theta(y_t \mid y_{<t}) & \text{if } t \in \text{response} \\ 0 & \text{if } t \in \text{prompt} \end{cases}$$


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example_chat(ex: Dict[str, str]) -> Tuple[str, str]:
    instruction = ex["instruction"].strip()
    inp = ex.get("input", "").strip()
    output = ex["output"].strip()

    user_content = instruction if not inp else f"{instruction}\n\n{inp}"
    messages = [{"role": "user", "content": user_content}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    return prompt_text, output

# Render the first training example
prompt_text, response_text = format_example_chat(train_rows[0])

print("=" * 60)
print("PROMPT (as the model sees it)")
print("=" * 60)
print(repr(prompt_text))
print()
print("=" * 60)
print("RESPONSE (the target — this is what gets gradient signal)")
print("=" * 60)
print(repr(response_text))


### Exercise 2: Why mask the prompt?

**2a.** *(Conceptual)* Suppose we did **not** mask the prompt tokens and trained on the full cross-entropy loss over the entire sequence. Describe one concrete way this could hurt the model. Think about what the model would be incentivized to learn, and whether that learning is useful at inference time.

**2b.** *(Conceptual)* The chat template inserts tokens like `<|user|>` and `<|assistant|>` around the content. These tokens are part of the "prompt" in our formulation and therefore get `label = -100`. Why is it important that the model still **attends to** them even though we don't compute loss on them?

**2c.** *(Conceptual)* In our dataset, every prompt is very similar (`"Translate this sentence into pirate speech."` plus a sentence). If we did *not* mask the prompt, what would the training loss look like after a few hundred steps on those nearly-identical prompt tokens? What does this suggest about the relationship between masking and the effective amount of supervision per example?


## 3. The Supervised Fine-Tuning Dataset

We wrap the formatting logic in a PyTorch `Dataset`. Each `__getitem__` call:

1. Formats the example with the chat template
2. Tokenizes *the prompt alone* to find out how long it is
3. Tokenizes the *full* `prompt + response + eos` sequence
4. Copies `input_ids` into `labels`, then overwrites the first `len(prompt)` positions with `-100`

The collator pads a batch to the longest sequence in it and similarly pads labels with `-100` (so padding doesn't contribute to loss either).


In [ ]:
class SFTDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]], tokenizer, max_length: int = 256):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ex = self.rows[idx]
        prompt_text, response_text = format_example_chat(ex)

        eos = self.tokenizer.eos_token or ""
        full_text = prompt_text + response_text + eos

        prompt_ids = self.tokenizer(
            prompt_text, add_special_tokens=False,
            truncation=True, max_length=self.max_length,
        )["input_ids"]

        full_enc = self.tokenizer(
            full_text, add_special_tokens=False,
            truncation=True, max_length=self.max_length,
        )
        input_ids      = full_enc["input_ids"]
        attention_mask = full_enc["attention_mask"]
        labels         = input_ids.copy()

        # Mask prompt tokens so loss applies only to the response
        prompt_len = min(len(prompt_ids), len(labels))
        for i in range(prompt_len):
            labels[i] = -100

        return {
            "input_ids":      torch.tensor(input_ids,      dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels":         torch.tensor(labels,         dtype=torch.long),
        }

train_ds = SFTDataset(train_rows, tokenizer)
val_ds   = SFTDataset(val_rows,   tokenizer)

# Peek at one tokenized example
sample = train_ds[0]
n_prompt = int((sample["labels"] == -100).sum().item())
n_response = int((sample["labels"] != -100).sum().item())
print(f"Total tokens:     {len(sample['input_ids'])}")
print(f"Masked (prompt):  {n_prompt}")
print(f"Scored (response):{n_response}")
print()
print("Decoded response portion (what the model is trained to produce):")
resp_ids = sample["input_ids"][sample["labels"] != -100]
print(" ", tokenizer.decode(resp_ids, skip_special_tokens=False))


## 4. Loading the Base Model and Generating a Baseline

Before we train anything, we should find out how well TinyLlama handles pirate translation *out of the box*. This gives us the baseline against which LoRA's improvement is measured, and it also sanity-checks our prompt formatting.


In [ ]:
from transformers import AutoModelForCausalLM

base_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=base_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)

print(f"Model loaded on: {next(model.parameters()).device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")


In [ ]:
@torch.no_grad()
def pirate_translate(sentence: str, model, max_new_tokens: int = 64,
                     do_sample: bool = False) -> str:
    messages = [{
        "role": "user",
        "content": f"Translate this sentence into pirate speech.\n\n{sentence}",
    }]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(completion, skip_special_tokens=True).strip()

TEST_SENTENCES = [
    "Please wait for your friends in the bar.",
    "Kira will bring the map to the market today.",
    "Can you find the window for the captain?",
    "I am looking for a quiet place to read my book.",  # out-of-template
]

print("=" * 60)
print("BASELINE (un-tuned TinyLlama)")
print("=" * 60)
for s in TEST_SENTENCES:
    print(f"\n> {s}")
    print(f"  {pirate_translate(s, model)}")


**Observation.** The base model has some pretty weird behavior - it often produces nonsensical stories that don't sound like pirates.  It also sees the phrase "translate" and thinks it means that it's supposed to translate to French or Spanish.  Very ineffective responses in my view.  

## 5. Injecting LoRA Adapters

Now we attach trainable low-rank adapters to the attention projection matrices of every transformer block. The `peft` library handles the mechanical work: it walks the model's module tree, finds every module whose name matches `target_modules`, and replaces each with a wrapped version that computes $W x + (\alpha/r) B A x$ — with $W$ frozen and $A, B$ trainable.

### Hyperparameters

- **`r`** — the rank of the adapter. Controls capacity: larger $r$ means more trainable parameters and more expressive updates, but also more risk of overfitting. Typical values: 4–64.
- **`lora_alpha`** — the scaling factor $\alpha$ in $\Delta W = (\alpha/r) BA$. Larger $\alpha$ amplifies the adapter's effect. A common heuristic is $\alpha = 2r$.
- **`lora_dropout`** — dropout applied to the input of $A$ during training, for regularization.
- **`target_modules`** — which linear layers to adapt. For Llama-family models, the attention projections `q_proj, k_proj, v_proj, o_proj` are the standard choice. Some recipes also adapt the MLP (`gate_proj, up_proj, down_proj`).


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


### Exercise 3: Parameter arithmetic

**3a.** *(Computation)* TinyLlama has 22 transformer blocks, each with 4 attention projection matrices (`q_proj, k_proj, v_proj, o_proj`). For this model the hidden dimension is 2048 and the key/value head dimension gives `k_proj` and `v_proj` the shape $(256, 2048)$ while `q_proj` and `o_proj` are $(2048, 2048)$. Using the LoRA formula $r \cdot (d_\text{in} + d_\text{out})$ per adapted matrix, compute by hand how many trainable parameters a rank-8 adapter on these four projection types should produce across all 22 blocks. Does it match the number `peft` reported?

**3b.** *(Conceptual)* Suppose you doubled the rank from $r=8$ to $r=16$. Roughly how does the number of trainable parameters change? How does the *number of floating-point operations in the LoRA forward pass* change? Is there any reason to think the base model's forward cost changes?

**3c.** *(Conceptual)* Why is it safe to initialize $B = 0$ at the start of training? What would break if we instead initialized both $A$ and $B$ with small random values?


## 6. Training

We use HuggingFace's `Trainer`. A few notes on the setup:

- **`DataCollatorForSeq2Seq`** pads variable-length sequences and, crucially, pads `labels` with `-100` (not 0) so padding tokens are excluded from the loss.
- **`learning_rate=2e-4`** is far higher than the ~1e-5 you'd use for full fine-tuning. LoRA can tolerate (and benefits from) larger learning rates because only a tiny number of parameters are moving and the base weights act as a regularizer.
- **`num_train_epochs=2`** is plenty for this task. On 2000 synthetic examples the LoRA adapter usually converges within one epoch.
- **`fp16=True`** if we have a GPU, halving memory and roughly doubling throughput.
- **`eval_strategy="epoch"`** runs validation loss at the end of each epoch so we can watch for overfitting.

Training takes ~2–3 minutes on a modern GPU, ~15–30 minutes on CPU.


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

OUT_DIR = Path("./pirate_lora_tinyllama")
OUT_DIR.mkdir(parents=True, exist_ok=True)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    use_cpu=(DEVICE == "cpu"),
    fp16=(DEVICE == "cuda"),
    bf16=False,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

trainer.train()

# Save the adapter (not the base model — just the tiny LoRA weights)
adapter_dir = OUT_DIR / "adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"\nAdapter saved to: {adapter_dir}")


### Exercise 4: Reading the training curves

**4a.** From the `Trainer` log above, record the training loss at step 20, step 100, and the final step, plus the two `eval_loss` values (one per epoch). Is the training loss going down monotonically? Is the eval loss?

**4b.** *(Conceptual)* If `eval_loss` were going *up* at the end of training while training loss kept falling, what would that tell you, and what would you do about it (concretely — what argument would you change)?

**4c.** *(Conceptual)* The saved "adapter" directory is only a few MB, compared to the ~2 GB TinyLlama checkpoint. List it (`!ls -lh pirate_lora_tinyllama/adapter`) and look at the actual file sizes. What makes it so small, and what does this imply about *serving* a fleet of dozens of specialized LoRAs on top of one shared base model?


## 7. Inference with the Trained Adapter

We can now generate with the LoRA-adapted model and compare directly against the baseline we recorded in §4. Because `peft` wraps the base model in place, our `pirate_translate` helper from earlier works unchanged — the same function now routes through the adapter on every forward pass.


In [ ]:
model.eval()

print("=" * 60)
print("AFTER LoRA FINE-TUNING")
print("=" * 60)
for s in TEST_SENTENCES:
    print(f"\n> {s}")
    print(f"  {pirate_translate(s, model)}")


### Exercise 5: Qualitative evaluation

**5a.** For each of the four test sentences, compare the baseline output (from §4) to the LoRA output above. In which cases is the improvement obvious? Is there a test sentence where LoRA produces a *worse* answer than the baseline?

**5b.** The fourth test sentence (`"I am looking for a quiet place to read my book."`) contains vocabulary that is **not** in our training templates (`"place"`, `"read"`, `"book"`). How does the LoRA model handle these out-of-vocabulary content words? Does it apply the learned stylistic substitutions (`the → th'`, etc.) to them anyway, or does it drop the style when it hits unfamiliar words?

**5c.** Write three English sentences of your own, run them through `pirate_translate(..., model)`, and evaluate the outputs. Try to find at least one sentence where the model fails in an interesting way (e.g., gets the style right but garbles the meaning, or vice versa). Explain *why* you think it failed.

**5d.** *(Conceptual)* LoRA updates attention projection matrices. The model's ability to adopt a *style* on top of otherwise-unchanged knowledge suggests something about how style is represented in a transformer. Speculate: why might rank-8 attention adapters be sufficient to install a stylistic rewrite, when the same rank would likely be insufficient to teach the model, say, a new language or a new domain of factual knowledge?


In [ ]:
# Exercise 5 — your code here


## 8. The Effect of Rank $r$: Capacity vs. Overfitting

The rank $r$ is LoRA's most important hyperparameter. Intuitively:

- **$r$ too small**: the adapter literally cannot represent the update $\Delta W$ the task needs. Training loss plateaus above the achievable minimum — classic *underfitting*.
- **$r$ too large**: the adapter has enough capacity to memorize individual training examples. On a small, low-diversity dataset like ours this shows up as training loss → 0 while validation loss starts rising — classic *overfitting*.
- **$r$ just right**: both losses fall together and the generated outputs feel stylistically consistent without echoing specific training sentences.

In this section you'll empirically find the sweet spot for our pirate task.


### Exercise 6: Rank sweep

**6a.** Re-run training with $r \in \{2, 8, 32\}$ (keeping `lora_alpha = 2*r` so the effective scale $\alpha/r$ stays fixed). For each rank, record:

- Number of trainable parameters
- Final train loss and final eval loss
- Qualitative output on the four test sentences

You can either (i) re-run the whole notebook three times and copy numbers, or (ii) refactor the training code into a `train_lora(r, alpha)` function and call it in a loop. The second option is cleaner but requires reloading the base model between runs (LoRA wraps the model in place, so you need a fresh base each time).

**6b.** *(Analysis)* Plot train and eval loss against rank on the same axes. At what rank, if any, do you see the train/eval gap widen? Does the qualitative output track the loss values, or does a rank with *worse* eval loss sometimes produce subjectively better pirate speak?

**6c.** *(Conceptual)* The LoRA paper reports that on GLUE tasks, rank $r = 1$ or $r = 2$ is often sufficient to match full fine-tuning. Why might the "right" rank be so small? Connect this to the paper's central hypothesis about the *intrinsic dimensionality* of task-specific updates. Would you expect the same tiny ranks to work for a much harder task, e.g., teaching the model to write Python code?

**6d.** *(Conceptual)* In our setup $\alpha/r$ is the scale applied to the adapter output. If you *increase* $r$ but keep $\alpha$ fixed (so $\alpha/r$ shrinks), what happens to the effective learning rate of the adapter? Why is the $\alpha = 2r$ convention a reasonable default, and what is it trying to hold constant across different ranks?


In [ ]:
# Exercise 6 — your code here


## 9. Scaling Up: What Changes in Real Fine-Tuning?

Everything you built in this lab — dataset formatting, prompt masking, LoRA injection, `Trainer` loop, adapter save/load — is structurally identical to what is used to fine-tune production LLMs. The differences are in scale, data quality, and a handful of refinements:

| Component | This lab | Production fine-tuning |
|-----------|----------|------------------------|
| **Base model** | TinyLlama 1.1B (fp16) | Llama 70B, Qwen2-72B, Mixtral 8×22B, often 4-bit quantized (QLoRA) |
| **Dataset** | 2000 synthetic templated examples | $10^5$–$10^7$ curated examples, carefully deduplicated and filtered for quality |
| **LoRA targets** | 4 attention projections | All linear layers (attention + MLP), sometimes embedding and lm_head too |
| **Rank** | Fixed $r = 8$ | Swept against held-out task metric; sometimes per-layer or learned (AdaLoRA, DoRA) |
| **Precision** | fp16 forward, fp32 optimizer state | 4-bit NF4 base weights + fp16 LoRA + paged AdamW (QLoRA); saves ~3× GPU memory |
| **Loss masking** | Prompt-vs-response binary mask | Same idea, plus multi-turn chat masking and sometimes token-level importance weighting |
| **Evaluation** | `eval_loss` on held-out val set | Downstream task metrics, LLM-as-judge, human preference comparisons, safety evals |
| **Serving** | Single adapter loaded on demand | Dozens of adapters hot-swapped per request (e.g., S-LoRA), shared base model in GPU memory |
| **Hardware** | 1 GPU or CPU | Multi-node clusters with FSDP / DeepSpeed, hours to days of wall time |

**The architecture is the same.** You've now touched every conceptual piece that separates a raw pretrained LLM from a specialized, instruction-following, domain-adapted version of it. The pirate corpus is small and silly; the machinery is exactly what runs inside modern LLM fine-tuning pipelines.
